In [27]:
!pip install catboost xgboost lightgbm optuna

In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [7]:
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


In [8]:
train.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [9]:
test_index = test['Index']

In [13]:
# Safe datetime conversion
train['timestamp'] = pd.to_datetime(
    train['timestamp'],
    errors='coerce'
)

test['timestamp'] = pd.to_datetime(
    test['timestamp'],
    errors='coerce'
)

# Fill invalid timestamps
train['timestamp'].fillna(
    train['timestamp'].mode()[0],
    inplace=True
)

test['timestamp'].fillna(
    test['timestamp'].mode()[0],
    inplace=True
)

print("Datetime conversion successful")

Datetime conversion successful


/tmp/ipykernel_514/2801948118.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  train['timestamp'] = pd.to_datetime(
/tmp/ipykernel_514/2801948118.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test['timestamp'] = pd.to_datetime(
/tmp/ipykernel_514/2801948118.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) in

In [14]:
for df in [train, test]:

    # hour
    df['hour'] = df['timestamp'].dt.hour

    # minute
    df['minute'] = df['timestamp'].dt.minute

    # weekday
    df['dayofweek'] = (
        df['timestamp'].dt.dayofweek
    )

    # month
    df['month'] = (
        df['timestamp'].dt.month
    )

    # week number
    df['weekofyear'] = (
        df['timestamp']
        .dt.isocalendar()
        .week
        .astype(int)
    )

    # weekend
    df['is_weekend'] = (
        df['dayofweek'] >= 5
    ).astype(int)

    # night
    df['is_night'] = (
        df['hour'].isin([0,1,2,3,4,5])
    ).astype(int)

    # rush hour
    df['peak_hour'] = (
        df['hour'].isin(
            [7,8,9,17,18,19]
        )
    ).astype(int)

In [15]:
for df in [train, test]:

    # hour cyclic
    df['hour_sin'] = np.sin(
        2*np.pi*df['hour']/24
    )

    df['hour_cos'] = np.cos(
        2*np.pi*df['hour']/24
    )

    # weekday cyclic
    df['dow_sin'] = np.sin(
        2*np.pi*df['dayofweek']/7
    )

    df['dow_cos'] = np.cos(
        2*np.pi*df['dayofweek']/7
    )

In [16]:
for df in [train, test]:

    # temperature × lanes
    df['temp_lane'] = (
        df['Temperature']
        * df['NumberofLanes']
    )

    # road + hour
    df['road_hour'] = (
        df['RoadType'].astype(str)
        + "_"
        + df['hour'].astype(str)
    )

    # geohash + hour
    df['geo_hour'] = (
        df['geohash'].astype(str)
        + "_"
        + df['hour'].astype(str)
    )

In [17]:
geo_mean = train.groupby(
    'geohash'
)['demand'].mean().reset_index()

geo_mean.columns = [
    'geohash',
    'geo_mean_demand'
]

train = train.merge(
    geo_mean,
    on='geohash',
    how='left'
)

test = test.merge(
    geo_mean,
    on='geohash',
    how='left'
)

In [18]:
geo_hour_mean = train.groupby(
    ['geohash', 'hour']
)['demand'].mean().reset_index()

geo_hour_mean.columns = [
    'geohash',
    'hour',
    'geo_hour_mean'
]

train = train.merge(
    geo_hour_mean,
    on=['geohash', 'hour'],
    how='left'
)

test = test.merge(
    geo_hour_mean,
    on=['geohash', 'hour'],
    how='left'
)

In [19]:
geo_hour_std = train.groupby(
    ['geohash', 'hour']
)['demand'].std().reset_index()

geo_hour_std.columns = [
    'geohash',
    'hour',
    'geo_hour_std'
]

train = train.merge(
    geo_hour_std,
    on=['geohash', 'hour'],
    how='left'
)

test = test.merge(
    geo_hour_std,
    on=['geohash', 'hour'],
    how='left'
)

In [20]:
train.fillna(-1, inplace=True)
test.fillna(-1, inplace=True)

In [21]:
drop_cols = [
    'timestamp',
    'Index'
]

X = train.drop(
    columns=drop_cols + ['demand']
)

y = train['demand']

X_test = test.drop(
    columns=drop_cols
)

In [22]:
cat_cols = [
    'geohash',
    'day',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',
    'road_hour',
    'geo_hour'
]

In [23]:
for col in cat_cols:

    X[col] = X[col].astype(str)

    X_test[col] = X_test[col].astype(str)

In [24]:
cat_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    depth=10,
    loss_function='RMSE',
    eval_metric='R2',
    verbose=500
)

In [25]:
cat_model.fit(
    X,
    y,
    cat_features=cat_cols
)

0:	learn: 0.0526003	total: 275ms	remaining: 13m 44s
500:	learn: 0.9625713	total: 2m 7s	remaining: 10m 34s
1000:	learn: 0.9695143	total: 4m 13s	remaining: 8m 25s
1500:	learn: 0.9739911	total: 6m 17s	remaining: 6m 17s
2000:	learn: 0.9772809	total: 8m 23s	remaining: 4m 11s
2500:	learn: 0.9796562	total: 10m 31s	remaining: 2m 6s
2999:	learn: 0.9816943	total: 12m 43s	remaining: 0us


CatBoostRegressor(depth=10, eval_metric='R2', iterations=3000, learning_rate=0.03, loss_function='RMSE', verbose=500)

In [28]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42
)

In [31]:
from sklearn.preprocessing import LabelEncoder

# Copy data
X_xgb = X.copy()
X_test_xgb = X_test.copy()

# Encode categorical columns
for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat([
        X_xgb[col],
        X_test_xgb[col]
    ]).astype(str)

    le.fit(combined)

    X_xgb[col] = le.transform(
        X_xgb[col].astype(str)
    )

    X_test_xgb[col] = le.transform(
        X_test_xgb[col].astype(str)
    )

# Train XGBoost
xgb_model.fit(X_xgb, y)

# Predict
pred_xgb = xgb_model.predict(
    X_test_xgb
)

In [30]:
lgb_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=10,
    random_state=42
)

In [34]:
# Train LightGBM
lgb_model.fit(X_xgb, y)

# Predict
pred_lgb = lgb_model.predict(
    X_test_xgb
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.047963 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1974
[LightGBM] [Info] Number of data points in the train set: 77299, number of used features: 20
[LightGBM] [Info] Start training from score 0.093942


In [40]:
# CatBoost prediction
pred_cat = cat_model.predict(X_test)

# XGBoost prediction
pred_xgb = xgb_model.predict(X_test_xgb)

# LightGBM prediction
pred_lgb = lgb_model.predict(X_test_xgb)

In [41]:
final_preds = (
    0.4 * pred_cat
    + 0.3 * pred_xgb
    + 0.3 * pred_lgb
)

In [38]:
submission = pd.DataFrame({
    'Index': test_index,
    'demand': final_preds
})

In [39]:
submission.to_csv(
    'submission.csv',
    index=False
)

print("submission.csv created")

submission.csv created
